# 03 Silver quality checks

Silver is clean and reshaped but nothing has been validated yet. This notebook is
the data-quality (DQ) layer: it runs a set of checks over the silver tables, records
the outcome of every check to **`dq_results`**, and copies any failing battles to
**`quarantine_battles`** for inspection.

These checks **record and continue**, they never stop the pipeline. So the data stays observable and the
dashboard can show how healthy each run was. They are **built for the dashboard.** `dq_results` is *appended* every run, so each tile can chart history. Completeness and freshness are stored as numbers in `metric_value`, so a tile can plot and threshold it directly.

## Config

In [0]:
from pyspark.sql import functions as F, types as T
from datetime import datetime

CATALOG, SCHEMA = "workspace", "clash"

# Sources (silver)
SILVER_BATTLES    = f"{CATALOG}.{SCHEMA}.silver_battles"
SILVER_DECK_CARDS = f"{CATALOG}.{SCHEMA}.silver_deck_cards"

# Targets (quality)
DQ_RESULTS          = f"{CATALOG}.{SCHEMA}.dq_results"
QUARANTINE_BATTLES  = f"{CATALOG}.{SCHEMA}.quarantine_battles"

# One id per run (a timestamp), so every row this run writes can be traced together.
RUN_ID = datetime.now().strftime("%Y%m%dT%H%M%S")
FRESHNESS_HOURS = 24   # flag if the newest battle is older than this
print("run", RUN_ID)

run 20260623T073427


## How a check works

Every check produces a small DataFrame of the **rows that fail** it (each carrying a
`battle_id`). `record()` then counts those rows, remembers the result for `quarantine`
checks to set the failing battles aside. `log` checks (like freshness) are recorded but
never quarantine anything.

Each recorded result carries a numeric **`metric_value`** for the dashboard, read per
check: failing-row count for quarantine checks, percentage for completeness, hours
for freshness.

In [0]:
results = []           # one row per check -> dq_results
quarantine_parts = []  # (battle_id, failed_check) frames to set aside

def record(name, failing, total, severity="error", action="quarantine", detail=None):
    """`failing` = the rows that FAIL this check (must contain battle_id when quarantining)."""

    # failed-row count appended to dq_results
    n = failing.count()
    results.append((RUN_ID, name, severity, action, n == 0, int(n), int(total), detail, float(n)))

    # failed rows quarantined
    if action == "quarantine" and n > 0:
        failed = failing.select("battle_id").distinct().withColumn("failed_check", F.lit(name))
        quarantine_parts.append(failed)

    status = "PASS" if n == 0 else "FAIL"
    print(f"[{status}] {name}: {n} of {total} failed")
    return n

## Battle-level checks

Run against `silver_battles` (one row per battle). Each `failing` frame is just the
battles that break the rule; an empty frame means the check passed.

Data is processed **incrementally** and appended to `dq_results` and `quarantine_battles`. Exit if no new data loaded from silver:

In [0]:
sb  = spark.table(SILVER_BATTLES)
sdc = spark.table(SILVER_DECK_CARDS)

sb_count = sb.count()
sdc_count = sdc.count()

# Get the incremental part of silver battles
if spark.catalog.tableExists(DQ_RESULTS):
    latest_timestamp = spark.table(DQ_RESULTS).select(F.max("ingested_at")).first()[0]
    delta_sb = sb.filter(F.col("ingested_at") > latest_timestamp)
else:
    delta_sb = sb

delta_sb_count = delta_sb.count()
print(f"{delta_sb_count} new battles, {sdc_count} deck cards to process")
if delta_sb_count == 0:
    dbutils.notebook.exit("No new data to process. Notebook stops at quality checks")

943043 new battles, 14213988 deck cards to process


In [0]:
# 1. battle_id is unique (verify the bronze's MERGE).
dup_ids = sb.groupBy("battle_id").count().filter("count > 1").select("battle_id")
record("battle_id_unique", dup_ids, sb_count)

# 2. battle_time is parsed cleanly, no null present.
bad_time = delta_sb.filter(F.col("battle_time").isNull()).select("battle_id")
record("battle_time_parsed", bad_time, delta_sb_count)

# 3. Crowns are in the valid 0-3 range, on both sides.
bad_crowns = delta_sb.filter(
    (F.col("player_crowns") < 0) | (F.col("player_crowns") > 3) |
    (F.col("opponent_crowns") < 0) | (F.col("opponent_crowns") > 3)
).select("battle_id")
record("crowns_in_range", bad_crowns, delta_sb_count)

# 4. Starting trophies are never negative (null is allowed — some modes don't report them).
bad_trophies = delta_sb.filter(
    (F.col("player_starting_trophies") < 0) | (F.col("opponent_starting_trophies") < 0)
).select("battle_id")
record("trophies_non_negative", bad_trophies, delta_sb_count)

# 5. Every deck has exactly 8 cards (size() of a null/short array != 8 -> flagged).
bad_deck = delta_sb.filter(
    (F.size("player_cards") != 8) | (F.size("opponent_cards") != 8)
).select("battle_id")
record("deck_has_8_cards", bad_deck, delta_sb_count)

[PASS] battle_id_unique: 0 of 943043 failed
[PASS] battle_time_parsed: 0 of 943043 failed
[FAIL] crowns_in_range: 5188 of 943043 failed
[PASS] trophies_non_negative: 0 of 943043 failed
[FAIL] deck_has_8_cards: 141161 of 943043 failed


141161

## Card library integrity

Run against `silver_deck_cards`. The silver join was a *left* join, so a card id that
isn't in `dim_cards` survives with a null `card_name` and will be flag here.

In [0]:
unknown_cards = sdc.filter(F.col("card_name").isNull()).select("battle_id").distinct()
record("card_id_in_dim", unknown_cards, sdc_count)

[PASS] card_id_in_dim: 0 of 14213988 failed


0

## Completeness — missing-data percentages

For the dashboard's **"% missing data"** tile. For each key field we count the rows where
it's NULL and store the **percentage** in `metric_value` (raw count stays in
`failed_count`). These are **log only** — a missing tag or trophy doesn't make the whole
battle unusable, so we report it rather than quarantine it.

`player_tag` / `opponent_tag` are the ones to watch: a missing tag means a deleted or
empty opponent, and (because `battle_id` is derived from the tags) too many of them can
weaken the uniqueness guarantee. Starting trophies are often legitimately missing in
non-ladder modes, so a non-zero % there is expected, not a bug.

In [0]:
def report_completeness(column, df, total):
    """Report what fraction of rows are missing (NULL in) `column`. Log only. metric_value holds the percentage, so the dashboard tile can read it directly."""
    missing = df.filter(F.col(column).isNull()).count()
    pct = 100 * missing / total if total else 0.0
    results.append((RUN_ID, f"missing_{column}", "warn", "log",
                    missing == 0, int(missing), int(total), f"{pct:.2f}% missing", float(pct)))
    print(f"[{'OK' if missing == 0 else 'WARN'}] missing_{column}: {missing} of {total} ({pct:.2f}%)")
    return missing

In [0]:
# Percentage of rows missing each key field (for the dashboard's "% missing data" tile).
# Tags first — a missing tag means a deleted/empty opponent and can weaken battle_id.
for column in ["battle_id", "player_tag", "opponent_tag",
               "player_starting_trophies", "opponent_starting_trophies"]:
    report_completeness(column, delta_sb, delta_sb_count)

[OK] missing_battle_id: 0 of 943043 (0.00%)
[OK] missing_player_tag: 0 of 943043 (0.00%)
[OK] missing_opponent_tag: 0 of 943043 (0.00%)
[WARN] missing_player_starting_trophies: 481249 of 943043 (51.03%)
[WARN] missing_opponent_starting_trophies: 458125 of 943043 (48.58%)


## Freshness

Not a row-level rule but a single signal of how recent the data is. The freshness is stored in
`metric_value` (hours) so the dashboard tile can plot it and colour it against the 24h threshold.
It is recorded but never quarantines anything.

In [0]:
latest = sb.agg(F.max("battle_time")).first()[0]
age_hours = (datetime.now() - latest).total_seconds() / 3600 if latest else None
is_fresh = age_hours is not None and age_hours < FRESHNESS_HOURS
detail = f"newest battle {age_hours:.1f}h ago" if age_hours is not None else "no battles"

# Logged directly (no failing rows to quarantine). metric_value = age in hours.
results.append((RUN_ID, "freshness_under_24h", "warn", "log", is_fresh,
                0 if is_fresh else 1, sb_count, detail,
                float(age_hours) if age_hours is not None else None))
print(f"[{'PASS' if is_fresh else 'WARN'}] freshness_under_24h: {detail}")

[WARN] freshness_under_24h: newest battle 132.4h ago


## Write results + quarantine

- **`dq_results`** — appended every run, so the dashboard can chart pass/fail and
  `metric_value` over time.
- **`quarantine_battles`** — overwritten each run with this run's failing battles (the
  full battle row, plus which check flagged it), for inspection.

In [0]:
# --- dq_results: one row per check, appended as run history ---
dq_schema = T.StructType([
    T.StructField("run_id", T.StringType()),
    T.StructField("check_name", T.StringType()),
    T.StructField("severity", T.StringType()),
    T.StructField("action", T.StringType()),
    T.StructField("passed", T.BooleanType()),
    T.StructField("failed_count", T.LongType()),
    T.StructField("total_count", T.LongType()),
    T.StructField("detail", T.StringType()),
    T.StructField("metric_value", T.DoubleType()),
])
dq_df = (spark.createDataFrame(results, dq_schema)
    .withColumn("checked_at", F.current_timestamp())
    .withColumn("ingested_at", F.lit(delta_sb.select(F.max("ingested_at")).first()[0])))
dq_df.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable(DQ_RESULTS)

# --- quarantine_battles: full rows of every flagged battle, overwritten per run ---
# Stack all the (battle_id, failed_check) frames into one. Start from an empty frame
# so this still works on a clean run where nothing failed.
flagged = spark.createDataFrame([], T.StructType([
    T.StructField("battle_id", T.StringType()),
    T.StructField("failed_check", T.StringType())]))
for part in quarantine_parts:
    flagged = flagged.unionByName(part)

quarantine = (flagged.join(sb, on="battle_id", how="left")
    .withColumn("run_id", F.lit(RUN_ID))
    .withColumn("quarantined_at", F.current_timestamp()))
(quarantine.write.format("delta").mode("append")
    .option("mergeSchema", "true").saveAsTable(QUARANTINE_BATTLES))

n_bad = flagged.select("battle_id").distinct().count()
print(f"{n_bad} distinct battles quarantined this run ({n_bad / delta_sb_count:.4%} of {delta_sb_count})")
display(dq_df.orderBy("action", "check_name").limit(10))

141161 distinct battles quarantined this run (14.9687% of 943043)


run_id,check_name,severity,action,passed,failed_count,total_count,detail,metric_value,checked_at,ingested_at
20260623T073427,freshness_under_24h,warn,log,false,1,943043,newest battle 132.4h ago,132.44879322916668,2026-06-23T07:36:26.287Z,2026-06-17T19:09:58.784558+00:00
20260623T073427,missing_battle_id,warn,log,true,0,943043,0.00% missing,0.0,2026-06-23T07:36:26.287Z,2026-06-17T19:09:58.784558+00:00
20260623T073427,missing_opponent_starting_trophies,warn,log,false,458125,943043,48.58% missing,48.57943911359291,2026-06-23T07:36:26.287Z,2026-06-17T19:09:58.784558+00:00
20260623T073427,missing_opponent_tag,warn,log,true,0,943043,0.00% missing,0.0,2026-06-23T07:36:26.287Z,2026-06-17T19:09:58.784558+00:00
20260623T073427,missing_player_starting_trophies,warn,log,false,481249,943043,51.03% missing,51.0315012146848,2026-06-23T07:36:26.287Z,2026-06-17T19:09:58.784558+00:00
20260623T073427,missing_player_tag,warn,log,true,0,943043,0.00% missing,0.0,2026-06-23T07:36:26.287Z,2026-06-17T19:09:58.784558+00:00
20260623T073427,battle_id_unique,error,quarantine,true,0,943043,null,0.0,2026-06-23T07:36:26.287Z,2026-06-17T19:09:58.784558+00:00
20260623T073427,battle_time_parsed,error,quarantine,true,0,943043,null,0.0,2026-06-23T07:36:26.287Z,2026-06-17T19:09:58.784558+00:00
20260623T073427,card_id_in_dim,error,quarantine,true,0,14213988,null,0.0,2026-06-23T07:36:26.287Z,2026-06-17T19:09:58.784558+00:00
20260623T073427,crowns_in_range,error,quarantine,false,5188,943043,null,5188.0,2026-06-23T07:36:26.287Z,2026-06-17T19:09:58.784558+00:00


## Done

Quality results are in `dq_results` and failing battles in `quarantine_battles`. The pipeline kept every clean row in silver untouched. Next: `04_gold_metrics`.